# Lekcja 1 — QPSK + AWGN

## Cel nauki
| Pojęcie | Co to jest | Rola w pipeline |
|---------|-----------|-----------------|
| **Bits** | 0/1 | Informacja źródłowa |
| **Mapper** | bits → symbole zespolone | Modulacja |
| **Konstelacja** | punkty w płaszczyźnie I/Q | Alfabet modulacji |
| **AWGN** | $y = x + n$ | Najprostszy kanał |
| **Demapper** | symbole → LLR | Soft detection |
| **BER** | odsetek błędnych bitów | Metryka jakości |

## Co zostanie zastąpione przez sieć?
W tej lekcji **nic** — to fundament. Sieć neuronowa w lekcji 7 zastąpi demapper (i wcześniejsze bloki OFDM), ale **LLR → LDPC** zostaje klasyczne.

## Równanie kanału
$$ y = x + n, \quad n \sim \mathcal{CN}(0, \sigma^2) $$


In [ ]:
import sys
from pathlib import Path

# Dodaj src/ do PYTHONPATH
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import sionna as sn
    import sionna.phy
except ImportError as e:
    raise ImportError(
        "Brak Sionny. Uruchom z katalogu magisterka/: ./scripts/drun sync"
    ) from e

from src.utils.setup import print_environment, get_device

sn.phy.config.seed = 42
device = get_device()
print_environment()


## Krok 1 — Źródło bitów

In [ ]:
NUM_BITS_PER_SYMBOL = 2  # QPSK = 2 bity/symbol
BATCH_SIZE = 1000
BLOCK_LENGTH = 1024

binary_source = sn.phy.mapping.BinarySource()
bits = binary_source([BATCH_SIZE, BLOCK_LENGTH])

print("Shape bits:", bits.shape)       # [batch, block_length]
print("Typ:", bits.dtype)
print("Przykład (pierwsze 8 bitów):", bits[0, :8].tolist())


## Krok 2 — Konstelacja i mapper

In [ ]:
constellation = sn.phy.mapping.Constellation("qam", NUM_BITS_PER_SYMBOL)
constellation.show()

mapper = sn.phy.mapping.Mapper(constellation=constellation)
# Mapper oczekuje BLOCK_LENGTH bitów podzielnych przez NUM_BITS_PER_SYMBOL
assert BLOCK_LENGTH % NUM_BITS_PER_SYMBOL == 0

x = mapper(bits)
print("Shape x (symbole TX):", x.shape)  # [batch, num_symbols]
print("Przykład symbolu:", x[0, 0])


## Krok 3 — Kanał AWGN

In [ ]:
awgn = sn.phy.channel.AWGN()
EBN0_DB = 10.0
no = sn.phy.utils.ebnodb2no(
    ebno_db=EBN0_DB,
    num_bits_per_symbol=NUM_BITS_PER_SYMBOL,
    coderate=1.0,  # brak kodowania
)

y = awgn(x, no)
print("Shape y:", y.shape)
print("Wariancja szumu no:", float(no))

from src.utils.plotting import plot_constellation
plot_constellation(x[:64], y[:64], title=f"QPSK @ Eb/N0={EBN0_DB} dB")
plt.show()


## Krok 4 — Demapper → LLR → hard bits

In [ ]:
demapper = sn.phy.mapping.Demapper("app", constellation=constellation)
llr = demapper(y, no)

print("Shape LLR:", llr.shape)  # taki sam jak bits
print("LLR > 0 → bit=1, LLR < 0 → bit=0")
bits_hat = (llr > 0).float()

from src.utils.metrics import ber
print(f"BER @ {EBN0_DB} dB:", ber(bits, bits_hat))


## Krok 5 — Krzywa BER vs SNR

In [ ]:
snr_range = np.arange(0, 12, 2)
ber_vals = []

for ebno_db in snr_range:
    no = sn.phy.utils.ebnodb2no(ebno_db, NUM_BITS_PER_SYMBOL, 1.0)
    b = binary_source([500, BLOCK_LENGTH])
    x = mapper(b)
    y = awgn(x, no)
    llr = demapper(y, no)
    ber_vals.append(ber(b, (llr > 0).float()))

from src.utils.plotting import plot_ber_curve
plot_ber_curve(snr_range, np.array(ber_vals), label="QPSK uncoded")
plt.show()


## Podsumowanie
- **Tensor bits**: `[batch, block_length]` — wartości 0/1
- **Tensor x**: `[batch, num_symbols]` — liczby zespolone
- **LLR** to logit dla każdego bitu — sieć neuronowa w lekcji 7 też produkuje LLR

## Ćwiczenie
Zmień `NUM_BITS_PER_SYMBOL` na 4 (16-QAM). Jak zmienia się BER przy tym samym Eb/N0?

**Następna lekcja:** `02_ofdm_basics.ipynb`
